# 因子构造 
**MA**因子构建    
取agent的行动 $action_{i,s,t}$  
其中，$i$ 为agent的编号，$s$ 为agent的行动，$t$ 为时间窗口     
对于每一个证券-时间窗口，求所有agent的行动的平均值，作为MA因子。     

对于不同的任务，都可以使用这个脚本生成MA因子   
任务包括基线、机制、稳健性（异质性分析使用基线数据）  
 

## 导入库

In [51]:
import json  
import os
import re
import polars as pl
from pathlib import Path
import warnings

## 超参数

TASK_ID_PREFIX: 用于指定任务ID，一般来说，分为baseline，mech，stable，表示基线，机制，稳健性  
异质性分析和基线使用同样数据 

In [ ]:
TASK_ID_PREFIX = 'baseline1'  # 任务id前缀
RESULTS_BASE_DIR = '/home/frank/files/programs/GraduationThesis/result' # 基本数据路径
SAVE_BASE_DIR = f'/home/frank/files/programs/GraduationThesis/empirical/{TASK_ID_PREFIX}' # 保存基本路径
SAVE = True # 是否保存数据


## 读取数据
数据的保存路径为：    

```
DATA_BASE_DIR  
|- TASK_ID1
|   |- Node1
|   |   |- performance_and_record_0.json   
|   |   |- performance_and_record_1.json
|   |   |- ...
|   |- Node2
|   |   |- performance_and_record_0.json
|   |   |- performance_and_record_1.json
|   |   |- ...
|   |- ...
|- TASK_ID2
|   |- Node1
|   |   |- performance_and_record_0.json
|   |   |- performance_and_record_1.json
|   |   |- ...
```  

其中，task_id的构成为 时间_中缀_uuid，例如：20260214_1924_lstm_short_66ab0230-9c11-4dc5-90e4-feb4b2c2fa57  
指定中缀，选取所有中缀一样的task_id，得到所有node的路径  

In [53]:
# 列出task_id下的所有no
pattern = re.compile(r'\d{8}_\d{4}_' + TASK_ID_PREFIX + r'_[a-z0-9\-]+')
matching_task_ids = [task_id for task_id in os.listdir(RESULTS_BASE_DIR) if pattern.match(task_id)] # 匹配中缀 

# 获取其下所有node的路径
node_paths = [] # 所有该task_id中缀的node路径
for task_id in matching_task_ids:
    node_paths.extend(
        os.path.join(
            RESULTS_BASE_DIR, task_id, 
            node_path
        )
        for node_path in os.listdir(os.path.join(RESULTS_BASE_DIR, task_id))
    ) 

len(node_paths)


283

In [54]:
# 获取其下所有performance_and_record_*.jsonl文件的路径
jsonl_paths = []
for node_path in node_paths:
    all_files = os.listdir(node_path)
    perf_and_rewa_jsonl_files = [file for file in all_files if file.endswith('.jsonl') and 'performance_and_reward_' in file]
    jsonl_paths.extend(
        os.path.join(node_path, file)
        for file in perf_and_rewa_jsonl_files
    )
len(jsonl_paths)

2500

先用scan获取所有的jsonl  
使用polars的concat方法，合并所有json，字段由performance_and_record约定好  
展示测试数据  

In [55]:
lazy_frames = [pl.scan_ndjson(f) for f in jsonl_paths]
lf = pl.concat(lazy_frames)   # 得到 LazyFrame


## 处理数据  
加载的lf，其结构如上所示，需要进行解析   

>- 1.基线回归为单证券，因此，需要验证portfolio中列表长度，如果有长度大于1的列表，需要警告，并且取[0]  
>- 2. data比较复杂，由多项fields组成，包括decision_weights, performance, normalized_performance, reward； 其中，如果是多证券，decision_weights, 是list结构，需要判断是否有长度大于1的列表，如果有，需要警告，并且取[0]; performance 和 normalized_performance是固定len=5的list，对应字段return, -vol, sharp, -maxdrawdwon, devisification，需要展开为对应字段   

In [56]:
perf_fields = ["return", "neg_vol", "sharp", "neg_maxdrawdown", "diversification"]

# 展开data
lf = lf.with_columns(pl.col("data").struct.unnest()).drop("data")

# 获取需要字段
lf = lf.select(
    pl.date(pl.col('year'),pl.col('month'),1).alias('date'),
    pl.col('portfolio').list.get(0).alias('portfolio'),
    pl.col('decision_weights').list.get(0).alias('decision_weights'),
    pl.col('performance').list.get(0).alias('return')
)

# 组合收益->证券收益
lf = lf.with_columns((pl.col('return') /(pl.col('decision_weights') + 1e-6)).alias('return'))

## 构建因子

去掉return为0的行  
（这些行往往是数据缺失） 

In [57]:
lf = lf.filter(pl.col('return').abs() >= 1e-4)

按照AED因子的定义，构建因子，按照year,month,portfolio进行分组，求每一个组decision_weights的 平均值，作为MA因子  

In [58]:
ma_lf = lf.group_by(['date','portfolio']).agg(
    (pl.col('decision_weights').mean()).alias('MA'),
    pl.col('return').first().alias('return')
)

## 保存数据  
保存数据为parquet文件  

In [59]:
if SAVE:
    dir_path = Path(SAVE_BASE_DIR)
    dir_path.mkdir(parents=True, exist_ok=True)
    ma_df = ma_lf.collect(engine='gpu')
    ma_df.write_parquet(os.path.join(SAVE_BASE_DIR, f'MA因子.parquet'))
    ma_df.write_parquet(os.path.join(SAVE_BASE_DIR, f'MA因子_copy.parquet'))
    print(f'保存成功，路径为:{os.path.join(SAVE_BASE_DIR, f'MA因子.parquet')}')
else:
    print('请设置SAVE=True，以保存数据')

ma_df.head()

保存成功，路径为:/home/frank/files/programs/GraduationThesis/empirical/baseline1/MA因子.parquet


date,portfolio,MA,return
date,str,f64,f64
2023-01-01,"""301019""",0.673006,0.0132
2022-11-01,"""002010""",0.811182,-0.069629
2017-03-01,"""600770""",0.607574,-0.1574
2024-03-01,"""300512""",0.839616,-0.0327
2006-11-01,"""002031""",0.649186,0.0552


In [60]:
del ma_df 
del ma_lf